# 모델 학습 실행 (SPEC-02)
런타임이 새로 뜰 때마다 2 → 3 → 4 순서로 실행한다. 5는 결과 확인용(읽기만 함).
- 학습이 끊기면 같은 순서로 다시 실행하면 이어서 학습한다 (SPEC-02 5절).
- 준비 순서와 파일 위치의 근거: SPEC-02 "확정 후 보완 3"
- 입력 파일은 임시 위치 `MyDrive/ai-image-detector-tmp/` (SPEC-02 미해결 8, 공유 위치 정해지면 셀 2만 바꾼다)
- 커밋 전에 출력을 모두 지운다 (편집 → 모든 출력 지우기)
- Colab 보안 비밀에 GITHUB_TOKEN이 있어야 한다
- 평가(SPEC-03)는 셀 2·3 다음에 평가 실행 → 평가 결과 확인 순서로 실행한다. 학습 설정은 셀 2에서, 평가 설정(val/test)은 평가 실행 칸의 EVAL_CONFIG에서 바꾼다. val로 metrics.csv와 맞는지 확인한 뒤 test를 돌린다.

In [2]:
# ===== 설정 (실험마다 이 셀만 바꾼다) =====
from pathlib import Path

CONFIG        = "src/model/configs/cifake_baseline.json"
DRIVE_DIR     = Path("/content/drive/MyDrive/ai-image-detector-tmp")  # 임시 위치 (SPEC-02 미해결 8)
ZIP_NAME      = "cifake.zip"
MANIFEST_NAME = "cifake_manifest.csv"
ZIP_SHA       = "cde8714fbe847e9b670be15b5b42a37b6b87f3f5481efc3032b0445351c54567"
MANIFEST_SHA  = "79e176eeb61bb024e6ffbc5163b5c84fa02f23daf74b9e39c2427086bcb3740c"
N_IMAGES      = 120000

ZIP_PATH  = Path("/content") / ZIP_NAME
DATA_ROOT = Path("/content/data")            # = config data_root
MANIFEST  = DATA_ROOT / MANIFEST_NAME        # = config manifest_path
REPO_URL  = "https://github.com/zeus1560/ai-image-detector.git"
REPO_DIR  = Path("/content/ai-image-detector")

In [3]:
# ===== 준비 (런타임마다 실행) =====
import os, time, json, shutil, subprocess

def sh(cmd):
    # 토큰이 들어가지 않는 명령에만 사용
    return subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True).stdout

def git_clone_private(repo_url, dest):
    """private 레포 clone. 토큰이 출력·예외 메시지에 나오지 않게 한다."""
    # 2. 위험한 위치(/content 자체, data, drive)를 지우지 않도록 막는다
    dest = Path(dest)
    assert dest.parent == Path("/content") and dest.name not in ("data", "drive"), f"clone 위치가 위험함: {dest}"
    # 3. https가 아니면 토큰이 들어가지 않아 원인 모를 실패가 나므로 미리 막는다
    assert repo_url.startswith("https://"), "repo_url은 https://로 시작해야 함"

    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    auth_url = repo_url.replace("https://", f"https://x-access-token:{token}@", 1)
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}   # 인증 실패 시 입력 대기하지 않음
    if dest.exists():
        shutil.rmtree(dest)                            # pull 대신 새로 clone
    # check=False: CalledProcessError(명령 문자열 포함)가 생기지 않는다
    # 1. stderr는 받아 두고, 실패하면 토큰을 가린 뒤 보여준다
    r = subprocess.run(["git", "clone", "-q", auth_url, str(dest)],
                       stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True, env=env)
    del auth_url
    if r.returncode != 0:
        err = (r.stderr or "").replace(token, "***") if token else (r.stderr or "")
        del token
        shutil.rmtree(dest, ignore_errors=True)        # 반쯤 만들어진 폴더 제거
        raise RuntimeError(f"clone 실패, 종료 코드 {r.returncode}: {err.strip()}")
    del token
    # clone 직후 remote URL을 토큰 없는 주소로 교체
    r = subprocess.run(["git", "-C", str(dest), "remote", "set-url", "origin", repo_url],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if r.returncode != 0:
        shutil.rmtree(dest, ignore_errors=True)        # 토큰이 남은 .git/config 제거
        raise RuntimeError(f"remote set-url 실패, 종료 코드 {r.returncode}")
    remote = sh(f"git -C {dest} remote get-url origin").strip()
    if "@" in remote:
        shutil.rmtree(dest, ignore_errors=True)
        raise RuntimeError("remote URL에 인증 정보가 남아 있어 폴더를 삭제함")
    return remote

# 0. GPU 확인
import torch
assert torch.cuda.is_available(), "GPU가 없습니다. 런타임 > 런타임 유형 변경에서 GPU를 선택하세요"
print("GPU:", torch.cuda.get_device_name(0))

# 1. Drive 연결 + zip 복사·확인
from google.colab import drive
drive.mount("/content/drive")
t = time.perf_counter()
shutil.copy(DRIVE_DIR / ZIP_NAME, ZIP_PATH)
assert sh(f"sha256sum {ZIP_PATH}").split()[0] == ZIP_SHA, f"{ZIP_NAME} sha256 불일치"
print(f"zip 복사·확인 {time.perf_counter()-t:.1f}s")

# 2. 압축 풀기 (DATA_ROOT를 새로 만듦)
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
t = time.perf_counter()
sh(f"unzip -q {ZIP_PATH} -d {DATA_ROOT}")
n_img = sum(1 for p in (DATA_ROOT / "images").rglob("*") if p.is_file())
assert n_img == N_IMAGES, f"이미지 수 {n_img} (기대 {N_IMAGES})"
print(f"unzip {n_img:,}개 {time.perf_counter()-t:.1f}s")

# 3. manifest 복사 (2번에서 DATA_ROOT를 지우므로 반드시 그 뒤에)
shutil.copy(DRIVE_DIR / MANIFEST_NAME, MANIFEST)
assert sh(f"sha256sum {MANIFEST}").split()[0] == MANIFEST_SHA, "manifest sha256 불일치"
print("manifest:", MANIFEST)

# 4. 레포 clone
remote = git_clone_private(REPO_URL, REPO_DIR)
print("레포:", remote, "@", sh(f"git -C {REPO_DIR} rev-parse --short HEAD").strip())

# 5. 설정 파일 경로가 위 준비 위치와 같은지 확인
cfg = json.loads((REPO_DIR / CONFIG).read_text(encoding="utf-8"))
assert Path(cfg["data_root"]) == DATA_ROOT, f"config data_root {cfg['data_root']} ≠ {DATA_ROOT}"
assert Path(cfg["manifest_path"]) == MANIFEST, f"config manifest_path {cfg['manifest_path']} ≠ {MANIFEST}"
print("config:", CONFIG, "run_name =", cfg["run_name"])
print("준비 완료")

GPU: Tesla T4
Mounted at /content/drive
zip 복사·확인 5.3s
unzip 120,000개 11.4s
manifest: /content/data/cifake_manifest.csv
레포: https://github.com/zeus1560/ai-image-detector.git @ daa0844
config: src/model/configs/cifake_baseline.json run_name = cifake_baseline
준비 완료


In [4]:
# ===== 학습 (끊기면 셀 2·3 후 이 셀을 다시 실행하면 이어서 학습) =====
!cd {REPO_DIR} && python -m src.model.train --config {CONFIG}

manifest 읽기: /content/data/cifake_manifest.csv (행 120000개, sha256=79e176eeb61bb024e6ffbc5163b5c84fa02f23daf74b9e39c2427086bcb3740c)
manifest 검사 통과 (행 120000개, origin_id 120000개)
  - split 겹침 origin_id 0개
  - label 불일치 origin_id 0개
  - test 전용 출처의 test 외 배정 0개
  - 파일 존재 검사: 전체 120000개, 없는 파일 0개
manifest 읽기: /content/data/cifake_manifest.csv (행 120000개, sha256=79e176eeb61bb024e6ffbc5163b5c84fa02f23daf74b9e39c2427086bcb3740c)
manifest 검사 통과 (행 120000개, origin_id 120000개)
  - split 겹침 origin_id 0개
  - label 불일치 origin_id 0개
  - test 전용 출처의 test 외 배정 0개
  - 파일 존재 검사: 전체 120000개, 없는 파일 0개
학습 시작: 2026-11-22 16:04:26
config: {"run_name": "cifake_baseline", "manifest_path": "/content/data/cifake_manifest.csv", "data_root": "/content/data", "source": "cifake", "crop_size": null, "model": "resnet50", "pretrained": true, "epochs": 5, "batch_size": 64, "lr": 0.0001, "weight_decay": 0, "seed": 42, "num_workers": 2, "output_dir": "/content/drive/MyDrive/ai-image-detector-runs"}
manifest_sha256: 79e17

In [5]:
# ===== 결과 확인: 실험 기록에 옮길 값 =====
run_dir = Path(cfg["output_dir"]) / cfg["run_name"]
for p in sorted(run_dir.iterdir()):
    print(f"{p.name:20s} {p.stat().st_size:>15,} B")
print()
print((run_dir / "metrics.csv").read_text(encoding="utf-8"))

best.pt                   94,357,887 B
config.json                      633 B
last.pt                  282,588,341 B
metrics.csv                      678 B
train_log.txt                  8,093 B

epoch,train_loss,val_loss,val_accuracy,val_false_positive_rate,val_false_negative_rate,val_ap,val_auc,epoch_seconds
1,0.16094527163505554,0.10207989832758904,0.9603,0.0389,0.0405,0.993689571796269,0.99358706,99.62686443328857
2,0.07643368526622653,0.08179694668650628,0.96835,0.0245,0.0388,0.9958086800076259,0.99586934,99.08282446861267
3,0.050370568564906716,0.09339009362608194,0.96645,0.0342,0.0329,0.9949861594889671,0.99499586,103.14259171485901
4,0.03772675567488186,0.08767875467762351,0.9698,0.0288,0.0316,0.9955705564232574,0.99578969,102.37064051628113
5,0.029432783001405186,0.08638750400915741,0.96965,0.0318,0.0289,0.9959158261788434,0.99571532,102.24646592140198



In [6]:
!head -c 300 /content/data/cifake_manifest.csv | od -c | head

0000000   p   a   t   h   ,   l   a   b   e   l   ,   s   o   u   r   c
0000020   e   ,   g   e   n   e   r   a   t   o   r   ,   o   r   i   g
0000040   i   n   _   i   d   ,   c   o   n   d   i   t   i   o   n   ,
0000060   s   p   l   i   t   ,   w   i   d   t   h   ,   h   e   i   g
0000100   h   t   ,   f   o   r   m   a   t   ,   c   o   l   o   r   _
0000120   m   o   d   e  \r  \n   i   m   a   g   e   s   /   c   i   f
0000140   a   k   e   /   o   r   i   g   i   n   a   l   /   f   a   k
0000160   e   /   c   i   f   a   k   e   _   f   a   k   e   _   t   e
0000200   s   t   _   0   _   1   .   j   p   g   ,   f   a   k   e   ,
0000220   c   i   f   a   k   e   ,   s   d   1   4   ,   c   i   f   a


In [11]:
# ===== 평가 (SPEC-03): 준비 셀 실행 후. val을 먼저 돌려 metrics.csv와 맞는지 본 뒤 test =====
EVAL_CONFIG = "src/model/configs/cifake_baseline_eval_test.json"

ecfg = json.loads((REPO_DIR / EVAL_CONFIG).read_text(encoding="utf-8"))
assert Path(ecfg["data_root"]) == DATA_ROOT, f"eval data_root {ecfg['data_root']} ≠ {DATA_ROOT}"
assert Path(ecfg["manifest_path"]) == MANIFEST, f"eval manifest_path {ecfg['manifest_path']} ≠ {MANIFEST}"
assert Path(ecfg["run_dir"]) == Path(cfg["output_dir"]) / cfg["run_name"], "평가 run_dir이 CONFIG의 학습 결과 폴더와 다름"
print("평가:", ecfg["eval_name"], "split =", ecfg["split"], "checkpoint =", ecfg["checkpoint"])
!cd {REPO_DIR} && python -m src.model.evaluate --config {EVAL_CONFIG}

평가: cifake_baseline_eval_test split = test checkpoint = best
manifest 읽기: /content/data/cifake_manifest.csv (행 120000개, sha256=79e176eeb61bb024e6ffbc5163b5c84fa02f23daf74b9e39c2427086bcb3740c)
manifest 검사 통과 (행 120000개, origin_id 120000개)
  - split 겹침 origin_id 0개
  - label 불일치 origin_id 0개
  - test 전용 출처의 test 외 배정 0개
  - 파일 존재 검사: 전체 120000개, 없는 파일 0개
평가 시작: 2026-11-22 16:16:44
평가 설정: {"eval_name": "cifake_baseline_eval_test", "run_dir": "/content/drive/MyDrive/ai-image-detector-runs/cifake_baseline", "checkpoint": "best", "manifest_path": "/content/data/cifake_manifest.csv", "data_root": "/content/data", "split": "test", "source": "cifake", "batch_size": 64, "num_workers": 2, "train_manifest_path": null}
학습 기록: model=resnet50, crop_size=None
체크포인트: /content/drive/MyDrive/ai-image-detector-runs/cifake_baseline/best.pt (epoch 2, sha256=b272083b6c6f94dbc34eb6a39a27926deee9ea17d9fdb36b497e9c7f8d0bd27e)
manifest_sha256: 79e176eeb61bb024e6ffbc5163b5c84fa02f23daf74b9e39c2427086bcb3740c
man

In [12]:
# ===== 평가 결과 확인: 실험 기록에 옮길 값 (완료 기준 B) =====
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

eval_dir = Path(ecfg["run_dir"]) / "eval" / ecfg["eval_name"]
rec = json.loads((eval_dir / "eval_config.json").read_text(encoding="utf-8"))
summary = pd.read_csv(eval_dir / "summary.csv")
pred = pd.read_csv(eval_dir / "predictions.csv")
print(f"{ecfg['eval_name']}: {rec['checkpoint']}.pt epoch {rec['checkpoint_epoch']}, predictions {len(pred):,}행")
print(pred["label"].map({0: "real", 1: "fake"}).value_counts().to_string())
print(summary.to_string(index=False))

# B3: AP·ROC-AUC를 sklearn으로 다시 계산해 비교 (두 라벨이 다 있는 그룹만)
cond = summary[summary["group"] == "condition"]
for (s, c), g in pred.groupby(["source", "condition"]):
    if g["label"].nunique() < 2:
        continue
    row = cond[(cond["source"] == s) & (cond["condition"] == c)].iloc[0]
    print(f"({s}, {c}) sklearn 대비 AP 차이 {abs(average_precision_score(g['label'], g['score']) - row['ap']):.2e}, "
          f"AUC 차이 {abs(roc_auc_score(g['label'], g['score']) - row['auc']):.2e}")

# B1: val 평가면 metrics.csv의 같은 epoch 값과 비교 (정확도·오탐률·미탐률은 차이 0, AP·AUC는 1e-4 이내)
if ecfg["split"] == "val":
    m = pd.read_csv(Path(ecfg["run_dir"]) / "metrics.csv").set_index("epoch").loc[rec["checkpoint_epoch"]]
    row = cond[cond["condition"] == "original"].iloc[0]
    for k in ["accuracy", "false_positive_rate", "false_negative_rate", "ap", "auc"]:
        theirs = m[f"val_{k}"]
        print(f"{k:20s} 평가 {row[k]:.6f}  metrics.csv {theirs:.6f}  차이 {abs(row[k] - theirs):.2e}")

cifake_baseline_eval_test: best.pt epoch 2, predictions 20,000행
label
fake    10000
real    10000
    group source condition generator  n_real  n_fake  n_false_positive  n_false_negative  accuracy  accuracy_ci_low  accuracy_ci_high  false_positive_rate  false_positive_rate_ci_low  false_positive_rate_ci_high  false_negative_rate  false_negative_rate_ci_low  false_negative_rate_ci_high       ap      auc
condition cifake  original       NaN 10000.0   10000             247.0               363    0.9695         0.967025          0.971794               0.0247                    0.021836                     0.027929               0.0363                    0.032809                     0.040148 0.995698 0.995733
generator cifake  original      sd14     NaN   10000               NaN               363       NaN              NaN               NaN                  NaN                         NaN                          NaN               0.0363                    0.032809                     0.040